# A1.5 · Multi-agent topology

**Function A — Security Architecture & Platform → The Security Architect**  ·  *Security of AI*

Builds on **[A1.4 · Blast radius as a design metric](https://spbreed.github.io/cyber-commons/lessons/A1.4.html)**.

| | |
|---|---|
| Open-source tooling | kagent |
| Open-weight models | GLM-4.6, Llama 3.3 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Multi-agent systems are sold on capability: a planner, a coder, a reviewer, each
good at one thing. What they actually introduce is **delegation depth**, and
depth is the variable nobody bounds.

Two properties change when one agent can call another:

**Authority composes.** A1.3 showed the walk-around: each grant legal, the
composition not. With three or four hops, no single reviewer sees the whole
path.

**Failure propagates.** If the orchestrator is compromised, every sub-agent is a
capability it now holds. The blast radius of the system is not the largest
agent's — it is the *sum over everything reachable*.

The topology decision is therefore a security decision, and the three common
shapes have genuinely different properties. This lesson measures them rather
than asserting a preference.

## 2 · Demo — three topologies, same capability

**Star**: one orchestrator calls specialised workers.
**Chain**: each agent hands to the next.
**Mesh**: any agent may call any other.

All three deliver "triage → fix → review → ship".

In [ ]:
SCOPE_WEIGHT = {"self": 1, "project": 3, "tenant": 8, "org": 20}
AGENTS = {   # agent -> (scope of its most dangerous tool, reversible?)
    "orchestrator": ("self",    True),
    "triager":      ("project", True),
    "fixer":        ("project", True),
    "reviewer":     ("project", True),
    "shipper":      ("org",     False),
}
def agent_blast(a):
    scope, rev = AGENTS[a]
    return SCOPE_WEIGHT[scope] * (1 if rev else 2)

TOPOLOGIES = {
    "star":  {"orchestrator": ["triager", "fixer", "reviewer", "shipper"],
              "triager": [], "fixer": [], "reviewer": [], "shipper": []},
    "chain": {"orchestrator": ["triager"], "triager": ["fixer"],
              "fixer": ["reviewer"], "reviewer": ["shipper"], "shipper": []},
    "mesh":  {a: [b for b in AGENTS if b != a] for a in AGENTS},
}

def reachable(topo, start):
    seen, stack = set(), [start]
    while stack:
        n = stack.pop()
        for m in topo.get(n, []):
            if m not in seen:
                seen.add(m); stack.append(m)
    return seen

def depth(topo, start, _seen=None):
    _seen = _seen or set()
    if start in _seen: return 0
    _seen = _seen | {start}
    kids = [depth(topo, k, _seen) for k in topo.get(start, []) if k not in _seen]
    return 1 + max(kids, default=0)

print(f"{'topology':10s}{'depth':>7}{'reachable from orchestrator':>32}{'blast':>8}")
print("-" * 60)
for name, topo in TOPOLOGIES.items():
    r = reachable(topo, "orchestrator")
    b = sum(agent_blast(a) for a in r)
    print(f"{name:10s}{depth(topo,'orchestrator'):>7}{len(r):>10} agents{'':<16}{b:>8}")

## 3 · Where it breaks

All three reach the same four agents and the same blast radius of 55, so on that metric alone they are identical. The difference shows up under **compromise**: which agent, if taken over, gives the attacker what?

In [ ]:
print(f"{'topology':10s}{'compromised':14s}{'reaches':>9}{'blast':>7}   worst case")
print("-" * 64)
for name, topo in TOPOLOGIES.items():
    worst, worst_b = None, -1
    for a in AGENTS:
        r = reachable(topo, a)
        b = sum(agent_blast(x) for x in r) + agent_blast(a)
        if b > worst_b:
            worst, worst_b = a, b
    for a in ("fixer", worst):
        r = reachable(topo, a)
        b = sum(agent_blast(x) for x in r) + agent_blast(a)
        tag = "  ← worst" if a == worst else ""
        print(f"{name:10s}{a:14s}{len(r):>9}{b:>7}{tag}")
    print()
print("In the mesh, compromising the LOWEST-privilege agent reaches everything.")
print("In the star, only the orchestrator does. That is the whole argument.")

## 4 · The control — bound the depth, and make it refuse

Two controls, both cheap:

1. **A depth limit**, enforced at the token issuer rather than the orchestrator — because when the orchestrator is the compromised component, a check inside it is worth nothing.
2. **No cycles.** A mesh where A can call B can call A has unbounded depth by construction.

In [ ]:
MAX_DEPTH = 3

class DepthExceeded(Exception): pass

def call(chain, callee, topo):
    if callee not in topo.get(chain[-1], []):
        raise PermissionError(f"{chain[-1]} may not call {callee} in this topology")
    if callee in chain:
        raise DepthExceeded(f"cycle: {' → '.join(chain)} → {callee}")
    if len(chain) + 1 > MAX_DEPTH:
        raise DepthExceeded(f"depth {len(chain)+1} > limit {MAX_DEPTH}: "
                            f"{' → '.join(chain)} → {callee}")
    return chain + [callee]

for topo_name, path in [("star",  ["orchestrator", "shipper"]),
                        ("chain", ["orchestrator", "triager", "fixer", "reviewer"]),
                        ("mesh",  ["orchestrator", "fixer", "orchestrator"])]:
    topo = TOPOLOGIES[topo_name]
    chain = [path[0]]
    try:
        for nxt in path[1:]:
            chain = call(chain, nxt, topo)
        print(f"{topo_name:6s} OK      {' → '.join(chain)}")
    except (DepthExceeded, PermissionError) as e:
        print(f"{topo_name:6s} REFUSED {e}")

## What you just proved

All three topologies reach 4 agents and a blast radius of 55. Under compromise they diverge sharply: in the mesh, taking over `fixer` reaches everything, while in the star only the orchestrator does. The depth limit permits the star's 2-hop call, refuses the chain at depth 4, and refuses the mesh cycle.

## Your turn

Draw your actual agent topology, including the calls that were added for convenience rather than design. If any two agents can call each other, you have a mesh — and the lowest-privilege agent in it is your real attack surface.

---

**Next → [A1.6 · Build vs buy](https://spbreed.github.io/cyber-commons/lessons/A1.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*